# Encoder–HistGB: Transformer optimization

The baseline remains the unchanged pipeline from **features_selection.ipynb**.
HistGB receives its existing features plus frozen encoder embeddings; only the encoder's
architecture is searched in this second pass. Inputs remain original imputed
features, missingness indicators and elapsed time, with bidirectional attention and no CLS token.

1. **Search:** 24 grid configurations, three patient splits, seed 42.
2. **Confirm:** best three configurations plus the incumbent, five patient splits, seeds 42, 43 and 44.
3. **Final model:** winning configuration, all training patients, a fixed seed.

Every fit keeps the internal 20% patient holdout for epoch selection, followed by a fresh
encoder fit on all fold-training patients. The objective is the complete Encoder–HistGB
validation MSE. There are no out-of-fold embeddings.

**Until rerun from the top, saved outputs below belong to the previous optimization.**

## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root or either notebook folder. N_TRIALS and TOP_K control the search and confirmation budgets.

In [11]:
import json
import os
import optuna
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)  # The reference setup reads data relative to the project.
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)  # Imputer helpers creation + one-hot → predictive imputation/flags → engineered features → median imputation/flags → no selection → tuned HistGB (unfitted).

# Start from the confirmed winner (trial 19) of the previous search.
ENCODER = dict(width=32, heads=8, layers=3, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 150, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
SEEDS = [42, 43, 44]
N_TRIALS, TOP_K = 24, 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. Build patients sequences (Encoder dataset)

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [12]:
def encoder_features(imputed):
    """Prepare per-visit inputs without patient IDs, adding elapsed time.

    Keep the original imputed features and missingness indicators. Compute
    elapsed years as age minus the patient's age at their first available visit.
    Patient IDs are used only for this grouping, not as encoder inputs.
    """
    
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    """Build one padded sequence per patient, with visits ordered by age.

    Features and targets must follow the same row order as raw. Return tensors
    of features [patients, visits, features], targets [patients, visits], and
    original row positions [patients, visits]. Padding positions are marked -1
    so attention and loss can ignore them; real positions restore embeddings
    to the original table order. Without targets, zero labels are unused
    placeholders for prediction.
    """
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Parameterized bidirectional encoder

Dense input projection → TransformerEncoder layers → one embedding per visit.
The grid varies joint width/head choices, encoder depth and regression-head depth. The head
contains two or three hidden layers; it supervises training but is not fed to HistGB.
Embedding width remains the Transformer width regardless of the regression head architecture.

In [13]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # 1. Initialize the encoder’s weights
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        # 2. Build the regression head
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [14]:
def train_network(training, seed, epochs, validation=None, params=None):
    torch.manual_seed(seed)
    params = {**ENCODER, "learning_rate": LEARNING_RATE,
              "weight_decay": WEIGHT_DECAY, **(params or {})}
    network = VisitEncoder(training.tensors[0].shape[-1],
                           **{key: params[key] for key in ENCODER}).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed, params=None):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping, params=params)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs, params=params)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [15]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Prepare the five patient splits once

The unchanged preprocessing and HistGB baseline are fitted once per split, then reused across
configurations. Search uses the first three splits; confirmation uses all five. Encoder early
stopping and refitting remain inside each fold and seed. No embeddings cross fold boundaries.

In [16]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        prepared_folds.append(dict(
            raw_train=raw_train, raw_val=raw_val, yt=yt, yv=yv,
            train=train, validation=validation,
            imputed_train=prep[:2].transform(raw_train), imputed_val=prep[:2].transform(raw_val),
            baseline_mse=mean_squared_error(yv, original.predict(validation)),
        ))


def evaluate_config(params, folds, seeds, trial=None):
    scores = []
    with threadpool_limits(limits=3):
        for fold_number, fold in enumerate(folds, 1):
            for seed in seeds:
                fitted = fit_encoder(fold["raw_train"], fold["yt"], fold["imputed_train"], seed, params)
                z_train, _ = encoder_outputs(fitted, fold["raw_train"], fold["imputed_train"])
                z_val, prediction = encoder_outputs(fitted, fold["raw_val"], fold["imputed_val"])
                model = clone(baseline[-1]).fit(fold["train"].join(z_train), fold["yt"])
                mse = mean_squared_error(fold["yv"], model.predict(fold["validation"].join(z_val)))
                scores.append(dict(fold=fold_number, seed=seed, epochs=fitted["epochs"],
                                   HistGB=fold["baseline_mse"], Encoder_HistGB=mse,
                                   Transformer=mean_squared_error(fold["yv"], prediction),
                                   delta_MSE=mse - fold["baseline_mse"]))
                if trial is None:
                    print(f"Fold {fold_number}/{len(folds)}, seed {seed}: MSE {mse:.4f}")
            if trial is not None:
                trial.report(np.mean([s["Encoder_HistGB"] for s in scores]), step=fold_number)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return pd.DataFrame(scores)

## 7. Optuna search: three splits, seed 42

Search the 24 grid configurations, evaluating the confirmed incumbent first. The objective
is mean **Encoder–HistGB MSE**, not standalone encoder MSE. Unpromising trials can be pruned
after at least two completed folds. N_TRIALS can cap the number of grid configurations tested.

Search space: joint (width, heads) choices (16, 8), (24, 8), (32, 8), (32, 16);
3/4/5 encoder layers; 2/3 regression-head hidden layers. All pairs satisfy width % heads == 0.
Keep ff_multiplier=3, dropout=0.1347886862599206, mlp_units=16,
learning_rate=0.001759366313629433 and weight_decay=4.3514588969246546e-05 fixed.
These parameters showed no clear directional boundary signal in the previous search.
The epoch ceiling is now 150 because one of the strongest previous trials reached 100;
patience remains 10 and batch size remains 128.


In [17]:
WIDTH_HEADS = {"16_8": (16, 8), "24_8": (24, 8),
               "32_8": (32, 8), "32_16": (32, 16)}
SEARCH_SPACE = dict(width_heads=list(WIDTH_HEADS), layers=[3, 4, 5], mlp_layers=[2, 3])


def encoder_params(params):
    """Decode grid choices and include the fixed architecture/training settings."""
    width, heads = WIDTH_HEADS[params["width_heads"]]
    return {**ENCODER, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
            "width": width, "heads": heads, "layers": params["layers"],
            "mlp_layers": params["mlp_layers"]}


def suggest_params(trial):
    return encoder_params({name: trial.suggest_categorical(name, values)
                           for name, values in SEARCH_SPACE.items()})


def objective(trial):
    scores = evaluate_config(suggest_params(trial), prepared_folds[:3], [42], trial)
    trial.set_user_attr("epochs", scores["epochs"].tolist())
    return scores["Encoder_HistGB"].mean()


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.BruteForceSampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
initial_params = dict(width_heads=f"{ENCODER['width']}_{ENCODER['heads']}",
                      layers=ENCODER["layers"], mlp_layers=ENCODER["mlp_layers"])
study.enqueue_trial(initial_params)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
study.trials_dataframe().sort_values("value")

/var/folders/q8/d21qf_z93f37d2z9b04pqw3h0000gn/T/ipykernel_54210/3768232637.py:26: ExperimentalWarning: BruteForceSampler is experimental (supported from v3.1.0). The interface can change in the future.
  direction="minimize", sampler=optuna.samplers.BruteForceSampler(seed=42),
[I 2026-11-03 12:37:59,438] A new study created in memory with name: no-name-d4398e60-f760-457a-b3ca-2833fce03b5f
Best trial: 0. Best value: 7.41769:   4%|▍         | 1/24 [03:00<1:09:20, 180.91s/it]

[I 2026-11-03 12:41:00,243] Trial 0 finished with value: 7.417688832723381 and parameters: {'width_heads': '32_8', 'layers': 3, 'mlp_layers': 2}. Best is trial 0 with value: 7.417688832723381.


Best trial: 0. Best value: 7.41769:   8%|▊         | 2/24 [06:47<1:16:10, 207.76s/it]

[I 2026-11-03 12:44:46,790] Trial 1 finished with value: 7.5023639534577455 and parameters: {'width_heads': '24_8', 'layers': 5, 'mlp_layers': 3}. Best is trial 0 with value: 7.417688832723381.


Best trial: 0. Best value: 7.41769:  12%|█▎        | 3/24 [08:52<59:28, 169.94s/it]  

[I 2026-11-03 12:46:51,767] Trial 2 finished with value: 7.548623504482797 and parameters: {'width_heads': '32_8', 'layers': 3, 'mlp_layers': 3}. Best is trial 0 with value: 7.417688832723381.


... (39 outputs left out)

[I 2026-11-03 13:59:14,004] Trial 22 pruned. 


Best trial: 9. Best value: 7.29054: 100%|██████████| 24/24 [1:26:43<00:00, 216.83s/it]

[I 2026-11-03 14:04:43,332] Trial 23 finished with value: 7.353866359847454 and parameters: {'width_heads': '32_16', 'layers': 4, 'mlp_layers': 2}. Best is trial 9 with value: 7.290535293582079.


,number,value,datetime_start,datetime_complete,duration,params_layers,params_mlp_layers,params_width_heads,user_attrs_epochs,system_attrs_fixed_params,state
9,9,7.290535,2026-11-03 13:05:20.886351,2026-11-03 13:12:09.104919,0 days 00:06:48.218568,5,2,24_8,"[53, 82, 77]",NaN,COMPLETE
20,20,7.299398,2026-11-03 13:45:20.880225,2026-11-03 13:52:38.980153,0 days 00:07:18.099928,4,2,16_8,"[74, 134, 71]",NaN,COMPLETE
10,10,7.353444,2026-11-03 13:12:09.177453,2026-11-03 13:17:21.256657,0 days 00:05:12.079204,5,2,32_8,"[33, 56, 54]",NaN,COMPLETE
23,23,7.353866,2026-11-03 13:59:14.116490,2026-11-03 14:04:43.331967,0 days 00:05:29.215477,4,2,32_16,"[60, 51, 44]",NaN,COMPLETE
5,5,7.399237,2026-11-03 12:53:55.562071,2026-11-03 12:56:57.575110,0 days 00:03:02.013039,3,2,32_16,"[68, 51, 52]",NaN,COMPLETE
0,0,7.417689,2026-11-03 12:37:59.444577,2026-11-03 12:41:00.243193,0 days 00:03:00.798616,3,2,32_8,"[56, 67, 86]","{'width_heads': '32_8', 'layers': 3, 'mlp_laye...",COMPLETE
1,1,7.502364,2026-11-03 12:41:00.353491,2026-11-03 12:44:46.790730,0 days 00:03:46.437239,5,3,24_8,"[88, 37, 48]",NaN,COMPLETE
4,4,7.537314,2026-11-03 12:51:19.960621,2026-11-03 12:53:55.460529,0 days 00:02:35.499908,4,3,32_8,"[37, 47, 39]",NaN,COMPLETE
2,2,7.548624,2026-11-03 12:44:46.903536,2026-11-03 12:46:51.767514,0 days 00:02:04.863978,3,3,32_8,"[41, 36, 46]",NaN,COMPLETE
18,18,7.558138,2026-11-03 13:37:37.269282,2026-11-03 13:40:44.408592,0 days 00:03:07.139310,5,2,32_16,NaN,NaN,PRUNED


## 8. Confirm the best configurations: five splits, three seeds

Evaluate the best TOP_K completed configurations (default three), plus the incumbent if
not already selected, on all five splits with seeds 42, 43 and 44. Select the configuration with the lowest mean pipeline MSE
across these 15 runs, not the seed with the lowest score. Standard deviations describe variability.
These scores guide model selection; because search and confirmation reuse validation patients,
they are not an independent, unbiased estimate of the selected model's test performance.

In [ ]:
candidates = sorted(
    (trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE),
    key=lambda trial: trial.value,
)[:TOP_K]
incumbent = study.trials[0]  # The confirmed configuration was enqueued first.
if all(candidate.number != incumbent.number for candidate in candidates):
    candidates.append(incumbent)
confirmation = []
for candidate in candidates:
    print(f"Confirming configuration from trial {candidate.number}")
    result = evaluate_config(encoder_params(candidate.params), prepared_folds, SEEDS)
    confirmation.append(result.assign(trial=candidate.number))
confirmation = pd.concat(confirmation, ignore_index=True)
confirmation_summary = confirmation.groupby("trial").agg(
    MSE=("Encoder_HistGB", "mean"), std=("Encoder_HistGB", lambda s: s.std(ddof=0)),
    HistGB=("HistGB", "mean"), delta_MSE=("delta_MSE", "mean"),
).sort_values("MSE")
winning_trial = int(confirmation_summary.index[0])
winning_params = next(encoder_params(trial.params) for trial in candidates
                      if trial.number == winning_trial)
scores = confirmation.loc[confirmation["trial"].eq(winning_trial)].copy()
print("Winning encoder parameters:", winning_params)
confirmation_summary

Confirming configuration from trial 9
Fold 1/5, seed 42: MSE 7.6435
Fold 1/5, seed 43: MSE 7.1083
Fold 1/5, seed 44: MSE 7.9541
Fold 2/5, seed 42: MSE 7.3280


In [ ]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.923423,0.352683,3.453031,0.000000
Transformer,8.429465,0.794165,2.903354,29.303310
Encoder_HistGB,7.477092,0.336742,2.734427,37.290727


## 9. Optional final training and test predictions

RUN_FINAL controls submission generation. The winning configuration is refitted on all training
patients with the fixed FINAL_SEED, retaining internal epoch selection and full-data refitting.
Only the encoder configuration changes; HistGB keeps its original parameters. Test patients
never fit preprocessing or either model. The optimized submission has a separate filename.

In [ ]:
RUN_FINAL = True
FINAL_SEED = 43

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    with threadpool_limits(limits=3):
        final_prep = clone(baseline[:-1])
        train = final_prep.fit_transform(X, y)
        test = final_prep.transform(X_test)
        imputed_train = final_prep[:2].transform(X)
        imputed_test = final_prep[:2].transform(X_test)
        final_encoder = fit_encoder(X, y, imputed_train, FINAL_SEED, winning_params)
        z_train, _ = encoder_outputs(final_encoder, X, imputed_train)
        z_test, _ = encoder_outputs(final_encoder, X_test, imputed_test)
        final_histgb = clone(baseline[-1]).fit(train.join(z_train), y)
        prediction = final_histgb.predict(test.join(z_test))
    assert np.isfinite(prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": prediction})
    submission_path = DATA_DIR / "submission_encoder_histgb_optimized.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")

Saved /Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/data/submission_encoder_histgb_optimized.csv (23,672 visits)


## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Replace the Transformer with another encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.